# AgriMitra AI demo

This notebook runs the farmer decision API: weather, crop choice, climate risk, mandi prices, and a short advisory.

Run the cells from top to bottom. AWS credentials are not required.

In [ ]:
import json
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "app.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from app import app

client = app.test_client()

def show(response):
    body = response.get_json()
    print("HTTP", response.status_code)
    print(json.dumps(body, indent=2, ensure_ascii=False)[:2500])
    return body

## 1. Weather for Nashik

Shows the current condition and whether spraying is suitable.

In [ ]:
weather = show(client.get("/api/weather?lat=19.9975&lon=73.7898&name=Nashik"))
current = weather["current"]
print("\nCondition:", current["condition"])
print("Temperature C:", current["temp_c"])
print("Spraying:", weather["climate_intelligence"]["spraying_suitability"])

## 2. What to grow

Kharif season, black soil, medium water, 2.5 acres.

In [ ]:
crops = show(client.post(
    "/api/crop/recommend",
    json={
        "season": "Kharif",
        "soil": "Black Soil",
        "water_availability": "Medium",
        "land_acres": 2.5,
    },
))
print("\nTop crops:")
for row in crops["top_recommendations"][:3]:
    print(f"- {row['name']}: score {row['suitability_score']}, net margin INR {row['est_net_margin_total']}")

## 3. Climate risk

Wheat during grain filling in Ludhiana, Punjab.

In [ ]:
risk = show(client.post(
    "/api/climate/risk-assessment",
    json={
        "crop": "Wheat",
        "growth_stage": "Grain / Pod Filling",
        "district": "Ludhiana",
        "state": "Punjab",
    },
))
score = risk["composite_risk"]
print("\nRisk:", score.get("level") or score.get("category"), "score", score.get("score"))
print("First action:", risk["emergency_protocols"]["pre_disaster_actions"][0])

## 4. Where to sell

A Nashik farmer with 60 quintals of onion.

In [ ]:
prices = show(client.get("/api/mandi/prices?crop=Onion&state=all"))
print("\nSample mandi:", prices["data"][0]["mandi_name"], "modal", prices["data"][0]["modal_price"])

sale = show(client.post(
    "/api/mandi/recommend-best",
    json={
        "farmer_lat": 19.9975,
        "farmer_lon": 73.7898,
        "farmer_location_name": "Nashik",
        "crop": "Onion",
        "quantity_qtl": 60,
    },
))
best = sale["best_mandi"]
print("\nBest mandi:", best.get("mandi_name") or best.get("name"))
print("Net profit INR:", best.get("net_profit"))

## 5. Farmer question

Uses Amazon Bedrock when `BEDROCK_MODEL_ID` is set. Otherwise it uses the local agronomy rules.

In [ ]:
chat = show(client.post(
    "/api/advisor/chat",
    json={
        "message": "How do I protect crops from unseasonal rainfall?",
        "language": "en",
        "farmer_id": "demo-farmer",
    },
))
print("\nPowered by:", chat.get("powered_by"))
print(chat.get("reply"))

## 6. AWS design

The deployed stack is in Sydney (`ap-southeast-2`).

In [ ]:
architecture = show(client.get("/api/aws/architecture"))
print("\nRegion:", architecture["region"])
for service in architecture["services"]:
    print("-", service["service"], ":", service["role"])